In [29]:
import numpy as np
import pandas as pd
import os
import zipfile


In [31]:

example_path = r'C:\Users\20242732\Desktop\cbl\cbl_predictive\multimodal-spectroscopic-dataset\data\example_data'

files = os.listdir(example_path)

first_file = os.path.join(example_path, files[0])
df = pd.read_parquet(first_file)


print("\nShape:", df.shape)
print("\nAll columns:")
for col in df.columns:
    print(f"  {col}")

print("\nFirst row (first column):")
print(df.iloc[0, 0])



Shape: (100, 21)

All columns:
  smiles
  hsqc_nmr_peaks
  hsqc_nmr_spectrum
  h_nmr_peaks
  h_nmr_spectra
  molecular_formula
  c_nmr_peaks
  ir_spectra
  msms_cfmid_positive_10ev
  msms_cfmid_positive_20ev
  msms_cfmid_positive_40ev
  msms_cfmid_fragments_positive
  msms_cfmid_negative_10ev
  msms_cfmid_negative_20ev
  msms_cfmid_negative_40ev
  msms_cfmid_fragments_negative
  c_nmr_spectra
  msms_iceberg_positive
  msms_iceberg_fragments_positive
  msms_scarf_positive
  msms_scarf_fragments_positive

First row (first column):
CC(C)CCNc1ncc(F)cc1C(=O)O


In [32]:
# Load only IR and SMILES columns
df = pd.read_parquet(
    example_path,
    columns=['smiles', 'ir_spectra']
)

print("Shape:", df.shape)
print(df.head(2))

Shape: (200, 2)
                         smiles  \
2228  CC(C)CCNc1ncc(F)cc1C(=O)O   
1784         Fc1ccc(-c2cccs2)s1   

                                             ir_spectra  
2228  [0.007766, 7.5e-05, 0.010152, 0.012034, 0.0336...  
1784  [0.001755, 0.017714, 0.018713, 0.021853, 0.006...  


In [33]:

zip_path = r'C:\Users\20242732\Desktop\cbl\cbl_predictive\4cblw010\MSS\mss_dataset\multimodal_spectroscopic_dataset.zip'
ir_only_path = r'C:\Users\20242732\Desktop\cbl\cbl_predictive\4cblw010\MSS\mss_dataset\ir_only'

os.makedirs(ir_only_path, exist_ok=True)

#Open zip and process parquet files one at a time w/o extracting everything to disk first
with zipfile.ZipFile(zip_path, 'r') as z:
    # Filter out __MACOSX files and only keep real parquet files
    parquet_files = [
        f for f in z.namelist() 
        if f.endswith('.parquet') and '__MACOSX' not in f
    ]
    print(f"Found {len(parquet_files)} real parquet files")

    for i, parquet_file in enumerate(parquet_files):
        print(f"Processing {i+1}/{len(parquet_files)}: {parquet_file}")

        try:
            with z.open(parquet_file) as f:
                df = pd.read_parquet(f, columns=['smiles', 'ir_spectra'])

            filename = os.path.basename(parquet_file)
            df.to_parquet(os.path.join(ir_only_path, filename))
            print(f"  Saved {len(df)} rows")

        except Exception as e:
            print(f"  Skipped — {e}")

print("\nDone")

total_rows = 0
for f in os.listdir(ir_only_path):
    if f.endswith('.parquet'):
        df = pd.read_parquet(os.path.join(ir_only_path, f))
        total_rows += len(df)
print(f"Total samples: {total_rows:,}")

print("\nDone. IR only data saved to:", ir_only_path)

# Check total size saved
total_rows = 0
for f in os.listdir(ir_only_path):
    df = pd.read_parquet(os.path.join(ir_only_path, f))
    total_rows += len(df)
print(f"Total samples available: {total_rows}")

BadZipFile: File is not a zip file

In [ ]:
from rdkit import Chem

ir_filtered_path = r'C:\Users\20242732\Desktop\cbl\cbl_predictive\4cblw010\MSS\mss_dataset\ir_filtered'

os.makedirs(ir_filtered_path, exist_ok=True)

# MARTS patterns
smarts = {
    'carboxylic_acid': Chem.MolFromSmarts('[CX3](=O)[OX2H1]'),
    'amino':           Chem.MolFromSmarts('[NX3H2]'),
    'sulfonic_acid':   Chem.MolFromSmarts('[$([#16X4](=[OX1])(=[OX1])([#6])[OX2H,OX1H0-]),$([#16X4+2]([OX1-])([OX1-])([#6])[OX2H,OX1H0-])]'),
    'guanidino':       Chem.MolFromSmarts('[$([NX3][CX3](=[NX2])[NX3]),$([NX3][CX3]([NX3])=[NX2])]'),
}

def get_labels(smiles_str):
    try:
        mol = Chem.MolFromSmiles(smiles_str)
        if mol is None:
            return None
        return {
            name: int(mol.HasSubstructMatch(pattern))
            for name, pattern in smarts.items()
        }
    except:
        return None

parquet_files = [f for f in os.listdir(ir_only_path) if f.endswith('.parquet')]
print(f"Processing {len(parquet_files)} files\n")

total_processed = 0
total_kept = 0

for i, filename in enumerate(parquet_files):
    print(f"File {i+1}/{len(parquet_files)}: {filename}")

    df = pd.read_parquet(os.path.join(ir_only_path, filename))
    total_processed += len(df)

    # Apply SMARTS labels
    labels = df['smiles'].apply(get_labels)

    # Drop invalid SMILES
    valid_mask = labels.notna()
    df = df[valid_mask].copy()
    labels = labels[valid_mask]

    # Expand label dict into columns
    label_df = pd.DataFrame(labels.tolist(), index=df.index)
    df = pd.concat([df, label_df], axis=1)

    # Keep only molecules with at least one functional group
    has_any = label_df.sum(axis=1) > 0
    df = df[has_any]
    total_kept += len(df)

    print(f"  {len(df)} molecules kept from {valid_mask.sum()} valid")

    if len(df) > 0:
        df.to_parquet(os.path.join(ir_filtered_path, filename))

print(f"\n{'='*50}")
print(f"Total processed: {total_processed:,}")
print(f"Total kept:      {total_kept:,}")
print(f"Reduction:       {(1 - total_kept/total_processed)*100:.1f}%")

# Load everything and show label distribution
all_df = pd.read_parquet(ir_filtered_path)
print(f"\nLabel distribution across {len(all_df):,} molecules:")
for col in ['carboxylic_acid', 'amino', 'sulfonic_acid', 'guanidino']:
    count = int(all_df[col].sum())
    pct = count / len(all_df) * 100
    print(f"  {col:<20}: {count:>7,} ({pct:.1f}%)")

Processing 245 files

File 1/245: aligned_chunk_0.parquet
  800 molecules kept from 3202 valid
File 2/245: aligned_chunk_1.parquet
  786 molecules kept from 3252 valid
File 3/245: aligned_chunk_10.parquet
  740 molecules kept from 3214 valid
File 4/245: aligned_chunk_100.parquet
  830 molecules kept from 3263 valid
File 5/245: aligned_chunk_101.parquet
  820 molecules kept from 3296 valid
File 6/245: aligned_chunk_102.parquet
  825 molecules kept from 3228 valid
File 7/245: aligned_chunk_103.parquet
  820 molecules kept from 3284 valid
File 8/245: aligned_chunk_104.parquet
  808 molecules kept from 3231 valid
File 9/245: aligned_chunk_105.parquet
  747 molecules kept from 3205 valid
File 10/245: aligned_chunk_106.parquet
  705 molecules kept from 3207 valid
File 11/245: aligned_chunk_107.parquet
  764 molecules kept from 3309 valid
File 12/245: aligned_chunk_108.parquet
  771 molecules kept from 3294 valid
File 13/245: aligned_chunk_109.parquet
  785 molecules kept from 3294 valid
File

: 

: 

In [30]:
# for final number of samples and label distribution

all_df = pd.read_parquet(ir_filtered_path)

print(f"Total samples: {len(all_df):,}")
print(f"\nLabel distribution:")
for col in ['carboxylic_acid', 'amino', 'sulfonic_acid', 'guanidino']:
    count = int(all_df[col].sum())
    pct = count / len(all_df) * 100
    print(f"  {col:<20}: {count:>7,} ({pct:.1f}%)")

NameError: name 'ir_filtered_path' is not defined